In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "utils"))

FIGURES_DIR = Path.cwd().parent / "report" / "figures"

from animals_utils import SUBLIMINAL_PROMPT_TEMPLATES, RELATION_MAP, SYNONYM_ANIMALS, agroups, get_animals_of_groups, get_numbers
from data_loading import (
    load_all_logprobs, make_combination, get_logprob_diff,
    prepare_relation_data, get_common_animals,
    compute_cosine_similarities, load_dataset_frequency_ratios,
    DEFAULT_RESPONSE_START, RELATION_ORDER,
)
from plotting import (
    scatter_logprob_vs_logprob, scatter_logprob_vs_logprob_mpl, scatter_single_animal,
    scatter_animal_vs_animal, scatter_logprob_vs_metric,
    heatmap_correlations_by_animal, heatmap_correlations_by_relation,
    plot_similarity_heatmap, get_animal_color_map,
)

results_dir = Path.cwd().parent / "results" / "Qwen2.5-7B-Instruct"
base_logprobs, subliminal_logprobs, unembedding_df = load_all_logprobs(results_dir)

animals = get_animals_of_groups([agroups.default])
print("Animals found:", animals)

In [ ]:
diff = lambda c: get_logprob_diff(c, subliminal_logprobs, base_logprobs)


love_love = make_combination("respondwithnumber", "love", "love")
love_love_withoutthinking = make_combination("withoutthinking", "love", "love")
love_hate_withoutthinking = make_combination("withoutthinking", "love", "hate")
hate_hate_withoutthinking = make_combination("withoutthinking", "hate", "hate")
love_adore = make_combination("respondwithnumber", "love", "adore")
love_hate = make_combination("respondwithnumber", "love", "hate")
love_detest = make_combination("respondwithnumber", "love", "detest")

scatter_logprob_vs_logprob_mpl(diff(love_love), diff(love_adore), love_love, love_adore, animals,
    note="Both axis are conditioned always answer with the number.<br>" \
    "X-axis is loving animals.<br>" \
    "Y-axis is adoring animals.<br>" \
    "The entangled (number, animal)-pairs are the same for both emotions.")

scatter_logprob_vs_logprob_mpl(diff(love_love), diff(love_hate), love_love, love_hate, animals,
    note="Both axis are conditioned to respond with the number.<br>" \
    "X-axis is loving animals.<br>" \
    "Y-axis is hating animals.<br>" \
    "In case of just responding with a number, entangled (number, animal)-pairs are the same for hate and love emotions." \
    "This is differen from 'loving a number'<br>" \
    "This was expected, if due to unembedding effects")

scatter_logprob_vs_logprob_mpl(diff(love_hate), diff(love_detest), love_hate, love_detest, animals,
    note="Both axis are conditioned to love the numbers.<br>" \
    "X-axis is hating animals.<br>" \
    "Y-axis is detesting animals.<br>")

In [ ]:
# Correlation heatmaps by animal
all_relation_data = prepare_relation_data(
    template_type="respondwithnumber",
    number_relation="love",
    subliminal_logprobs=subliminal_logprobs,
    base_logprobs=base_logprobs,
    baseline="default",
)
print("Available relations:", list(all_relation_data.keys()))

synonym_animal_names = {a[0] for a in SYNONYM_ANIMALS}
non_synonym_animals = [a for a in get_common_animals(all_relation_data) if a not in synonym_animal_names]

heatmap_correlations_by_animal(
    all_relation_data,
    title="Correlation Matrix per Animal: Logprob Differences for Different Animal Relations\n(Conditioned: Love without thinking about a number)",
    relation_order=RELATION_ORDER,
    animals=non_synonym_animals,
)

In [ ]:
scatter_logprob_vs_logprob_mpl(diff(love_love), diff(love_love_withoutthinking), love_love, love_love_withoutthinking, animals,
    note="" \
    "X-axis is loving animals (with number focus).<br>" \
    "Y-axis is loving animals (without number focus).<br>" \
    "The numbers, that steer when loving them are correlated with the numbers that steer when always responding with them.<br>" \
    "The correlation is notably weaker, then e.g. between ")

scatter_logprob_vs_logprob_mpl(diff(love_love), diff(hate_hate_withoutthinking), love_love, hate_hate_withoutthinking, animals,
    note="Both axis are conditioned to love the numbers.<br>" \
    "X-axis is loving animals (with number focus).<br>" \
    "Y-axis is hating animals (without number focus).<br>" \
    "TODO" \
    "TODO")

In [ ]:
# Animal-level analysis: average animal-animal correlation vs unembedding cosine
from data_loading import build_animal_pair_correlation_cosine_df
from plotting import heatmap_correlations_by_relation, heatmap_correlations_by_relation_export, scatter_relation_pair_avg_r_vs_unembedding_cosine, scatter_relation_pair_avg_r_vs_unembedding_cosine_export
from animals_utils import SINGLE_TOKEN_RELATIONS_QWEN, SINGLE_TOKEN_ANIMALS_QWEN

single_token_animals = [
    animal for animal in SINGLE_TOKEN_ANIMALS_QWEN
    if animal in get_common_animals(all_relation_data)
]


selected_animals = ["rabbit", "bunny", "hare", "mosquito", "cockroach", "cat", "kitty", "dog", "canine","panda", "kangaroo", "koala", "lion"]
print("Single-token animals used:", single_token_animals)
print("selected animals used:", selected_animals)

animal_pair_df = build_animal_pair_correlation_cosine_df(
    all_relation_data=all_relation_data,
    unembedding_df=unembedding_df,
    relations=SINGLE_TOKEN_RELATIONS_QWEN,
    animals=single_token_animals,
)

pairs_to_label = [("mouse", "rat"), ("mouse", "rabbit"), ("dog", "cat"), ("bunny", "cougar"), ("hog", "cattle"), ("rabbit", "bunny")]

scatter_relation_pair_avg_r_vs_unembedding_cosine(
    animal_pair_df,
    x_axis_text="Average Pearson r across relations (animal pair)",
    y_axis_text="Cosine similarity\n(unembedding vectors)",
    label_pairs=pairs_to_label,
)

scatter_relation_pair_avg_r_vs_unembedding_cosine_export(
    animal_pair_df,
    x_axis_text="Average Pearson r across relations (animal pair)",
    y_axis_text="Cosine similarity\n(unembedding vectors)",
    export_path=FIGURES_DIR / "animal_pair_avg_r_vs_unembedding_cosine_respondwithnumber.png",
    label_pairs=pairs_to_label,
)

In [ ]:
# Scatter: average relation-pair correlation (r) vs unembedding cosine similarity
from data_loading import build_relation_pair_correlation_cosine_df
from plotting import scatter_relation_pair_avg_r_vs_unembedding_cosine, scatter_relation_pair_avg_r_vs_unembedding_cosine_export
from animals_utils import SINGLE_TOKEN_RELATIONS_QWEN

pair_df = build_relation_pair_correlation_cosine_df(
    all_relation_data=all_relation_data,
    unembedding_df=unembedding_df,
    relations=SINGLE_TOKEN_RELATIONS_QWEN,
    animals=non_synonym_animals,
 )

pairs_to_label = [("hate", "dislike"), ("like", "cherish"), ("love", "hate"), ("like", "scorn"), ("tolerate", "reject")]

display(pair_df)
scatter_relation_pair_avg_r_vs_unembedding_cosine(
    pair_df,
    x_axis_text="Average Pearson r across animals (relation pair)",
    y_axis_text="Cosine similarity (relation unembedding vectors)",
    label_pairs=pairs_to_label,
)
scatter_relation_pair_avg_r_vs_unembedding_cosine_export(
    pair_df,
    x_axis_text="Average Pearson r across animals",
    y_axis_text="Cosine similarity\n(unembedding vectors)",
    export_path=FIGURES_DIR / "relation_pair_avg_r_vs_unembedding_cosine_respondwithnumber.png",
    label_pairs=pairs_to_label,
)

In [ ]:
# Scatter: average relation-pair correlation (r) vs unembedding cosine similarity
from data_loading import build_relation_pair_correlation_cosine_df
from plotting import scatter_relation_pair_avg_r_vs_unembedding_cosine, scatter_relation_pair_avg_r_vs_unembedding_cosine_export
from animals_utils import SINGLE_TOKEN_RELATIONS_QWEN

pair_df = build_relation_pair_correlation_cosine_df(
    all_relation_data=all_relation_data,
    unembedding_df=unembedding_df,
    relations=SINGLE_TOKEN_RELATIONS_QWEN,
    animals=non_synonym_animals,
 )

pairs_to_label = [("hate", "dislike"), ("like", "cherish"), ("love", "hate"), ("like", "scorn"), ("tolerate", "reject")]

display(pair_df)
scatter_relation_pair_avg_r_vs_unembedding_cosine(
    pair_df,
    x_axis_text="Average Pearson r across animals (relation pair)",
    y_axis_text="Cosine similarity (relation unembedding vectors)",
    label_pairs=pairs_to_label,
)
scatter_relation_pair_avg_r_vs_unembedding_cosine_export(
    pair_df,
    x_axis_text="Average Pearson r across animals",
    y_axis_text="Cosine similarity\n(unembedding vectors)",
    export_path=FIGURES_DIR / "relation_pair_avg_r_vs_unembedding_cosine_respondwithnumber.png",
    label_pairs=pairs_to_label,
)